# Day 6: Tools and Your First Agent

Our chatbot from yesterday can talk, but that's all it can do. Ask it "what's 4,739 times 286?" and it might get it wrong, because language models are guessing the next word, not doing arithmetic. Ask it today's weather and it simply doesn't know.

Today we fix that by giving the model **tools**.

Here's the analogy I use in class. Imagine a very smart manager who can't leave the office. The manager can't check the warehouse or open the accounts software, but they can send an assistant: "go check how many boxes we have and tell me". The manager decides *what* needs doing; the assistant actually *does* it and reports back.

The LLM is the manager. Your Python functions are the assistants. The model never runs your code itself. It just says "please run this function with these inputs", your program runs it, and you hand the result back.

When the model can keep doing that in a loop (think, use a tool, look at the result, think again) until the job is done, you have an **agent**.

By the end of today you'll be able to:

- turn a Python function into a tool
- see the model ask for a tool
- run that tool and send back the result yourself, by hand
- build the agent loop as a graph
- use LangGraph's ready-made pieces to do the same thing with less code

In [ ]:
%pip install -q -U langgraph langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. What a tool actually is

A tool is a normal Python function with the `@tool` decorator on top. That's really all.

But here's the part people miss: **the model never sees your code.** It only sees three things:

- the function's **name**
- its **arguments and their types**
- its **docstring**

So the docstring isn't documentation for other programmers any more. It's the instruction manual the model reads to decide when to use the tool. A vague docstring means the model will use the tool at the wrong time, or never.

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers. Use this for any multiplication."""
    return a * b

In [ ]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

Let's look at what the model will actually see.

In [ ]:
print("name:       ", multiply.name)
print("description:", multiply.description)
print("arguments:  ", multiply.args)

A tool is still a function, so you can call it yourself. Tools take their arguments as a dictionary:

In [ ]:
print(multiply.invoke({"a": 12, "b": 7}))

## 2. The model asks for a tool

To let the model know which tools exist, we use `bind_tools`. Think of it as handing the manager a list of assistants and what each one can do.

In [ ]:
tools = [multiply, add]
llm_with_tools = llm.bind_tools(tools)

### Predict before you run

We'll ask "What is 123 multiplied by 45?". Will `.content` contain the answer 5535?

<details>
<summary>Click to see the answer</summary>

No. `.content` will most likely be empty.

Instead of answering, the model fills in `.tool_calls` with a request, something like `[{'name': 'multiply', 'args': {'a': 123, 'b': 45}, ...}]`. It's saying "I need you to run multiply for me first." Nothing has been calculated yet.
</details>

In [ ]:
ai_message = llm_with_tools.invoke("What is 123 multiplied by 45?")

print("content:   ", repr(ai_message.content))
print("tool_calls:", ai_message.tool_calls)

And if the question doesn't need a tool, the model just answers normally. It decides for itself.

In [ ]:
plain = llm_with_tools.invoke("Say good morning in Hindi.")

print("tool_calls:", plain.tool_calls)
print("content:   ", plain.content)

## 3. Running the tool by hand

Before we let LangGraph automate anything, let's do the full round trip ourselves once. It's like learning to cook a dish by hand before using the food processor: you understand what the machine is doing.

The round trip has four steps:

1. The user asks a question.
2. The model replies with `tool_calls`.
3. **We** run the tool, and send the result back as a `ToolMessage`. The `tool_call_id` must match the request, so the model knows which answer belongs to which question.
4. The model reads the result and writes the final answer.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
# A quick way to find a tool by its name
tools_by_name = {t.name: t for t in tools}

In [ ]:
# Steps 1 and 2: ask, and get the tool request back
messages = [HumanMessage(content="What is 123 multiplied by 45?")]

ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [ ]:
# Step 3: run each requested tool and attach the result
for call in ai_message.tool_calls:
    chosen_tool = tools_by_name[call["name"]]
    output = chosen_tool.invoke(call["args"])
    print(f"Ran {call['name']} with {call['args']} -> {output}")

    messages.append(ToolMessage(content=str(output), tool_call_id=call["id"]))

In [ ]:
# Step 4: the model sees the result and answers
final = llm_with_tools.invoke(messages)
print(final.content)

That's honestly the whole trick behind "AI that uses tools". Everything else is just automating this loop.

## 4. Building the agent loop yourself

Some questions need more than one tool call. "What is (3 + 5) times 12?" needs `add` first, and only then `multiply`. So we can't do one round trip and stop. We need to **loop until the model stops asking for tools.**

Look at that sentence again, because it's exactly what we learned on Day 3: a node, a router, and an edge pointing backwards.

```
  START --> agent --(asked for tools?)--yes--> run_tools --+
              ^                                            |
              +--------------------------------------------+
              |
              no
              v
             END
```

This pattern has a name, **ReAct** (Reason + Act), and it's the foundation of almost every agent you'll see.

In [ ]:
from typing import Literal
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState

In [ ]:
SYSTEM = SystemMessage(content="You are a maths helper. Always use the tools for arithmetic, never calculate in your head.")

In [ ]:
def agent(state: MessagesState) -> dict:
    # The "thinking" step: look at the conversation and decide what to do next
    reply = llm_with_tools.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

In [ ]:
def run_tools(state: MessagesState) -> dict:
    # The "doing" step: run whatever the model asked for
    last_message = state["messages"][-1]
    results = []

    for call in last_message.tool_calls:
        output = tools_by_name[call["name"]].invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))

    return {"messages": results}

In [ ]:
def should_continue(state: MessagesState) -> Literal["run_tools", "__end__"]:
    # If the model asked for tools, go run them. Otherwise it has answered, so stop.
    if state["messages"][-1].tool_calls:
        return "run_tools"
    return END

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", agent)
builder.add_node("run_tools", run_tools)

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("run_tools", "agent")     # this is the loop: after tools, think again

In [ ]:
my_agent = builder.compile()
show_graph(my_agent)

### Predict before you run

For "What is (3 + 5) multiplied by 12?", how many times do you think the `agent` node will run?

<details>
<summary>Click to see the answer</summary>

Usually three times:

1. It asks for `add(3, 5)`.
2. It sees 8, and asks for `multiply(8, 12)`.
3. It sees 96, and writes the final answer.

Sometimes a model is clever and asks for both tools at once, but since `multiply` needs the result of `add`, it typically goes one step at a time. Look at the printed messages to see what your model did.
</details>

In [ ]:
result = my_agent.invoke({"messages": [("user", "What is (3 + 5) multiplied by 12?")]})

for m in result["messages"]:
    m.pretty_print()

## 5. The same agent, with less code

This pattern is so common that LangGraph ships the two pieces ready-made:

- **`ToolNode(tools)`** does exactly what our `run_tools` did. It even runs several tool calls at the same time.
- **`tools_condition`** does exactly what our `should_continue` did. It goes to a node called `"tools"` if there are tool calls, otherwise to END.

One small catch: `tools_condition` expects the tool node to be named `"tools"`.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))    # the name must be "tools"

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

In [ ]:
agent_app = builder.compile()
show_graph(agent_app)

Let's watch it work step by step with `stream_mode="updates"` (from Day 1), so we can see each tool request and each result as it happens.

In [ ]:
question = {"messages": [("user", "Add 2.5 and 7.5, then multiply the result by 4")]}

for step in agent_app.stream(question, stream_mode="updates"):
    for node_name, update in step.items():
        for message in update["messages"]:
            if getattr(message, "tool_calls", None):
                print(f"[{node_name}] wants to call: {message.tool_calls}")
            else:
                print(f"[{node_name}] {message.type}: {message.content}")

## 6. When a tool fails

Tools call real things, and real things fail. Someone divides by zero, an API times out.

By default, an exception inside a tool **stops the whole graph**. Usually that's not what you want. You'd rather tell the model "that didn't work, here's why" and let it recover. `ToolNode(tools, handle_tool_errors=True)` does exactly that: the error comes back to the model as a `ToolMessage` instead of crashing.

In [ ]:
from langchain_core.messages import AIMessage

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("tools", ToolNode([divide], handle_tool_errors=True))

In [ ]:
builder.add_edge(START, "tools")
builder.add_edge("tools", END)

In [ ]:
safe_tools = builder.compile()

In [ ]:
# Let's pretend the model asked us to divide by zero
bad_request = AIMessage(
    content="",
    tool_calls=[{"name": "divide", "args": {"a": 1, "b": 0}, "id": "call_1"}],
)

result = safe_tools.invoke({"messages": [bad_request]})
print(result["messages"][-1].content)

No crash. The model would receive that error text and could reply "Sorry, you can't divide by zero" instead of your app falling over.

## 7. The one-line agent: `create_agent`

Now that you know what's inside an agent, here's the shortcut. LangChain's `create_agent` builds the same agent-and-tools loop for you. And since it returns a LangGraph graph, everything you've learned (`invoke`, `stream`, and the memory we add tomorrow) works on it.

In [ ]:
from langchain.agents import create_agent

In [ ]:
quick_agent = create_agent(
    model=llm,
    tools=[multiply, add, divide],
    system_prompt="You are a helpful maths assistant. Use the tools for every calculation.",
)

In [ ]:
show_graph(quick_agent)

In [ ]:
result = quick_agent.invoke({"messages": [("user", "What is 144 divided by 12, plus 10?")]})
print(result["messages"][-1].content)

So when should you use which? Learn with the hand-built version (sections 4 and 5). Use `create_agent` when you need a standard agent quickly. Build your own graph when you need custom steps, extra routing, or several agents working together, which is what we'll be doing from Day 10 onwards.

## 8. Common mistakes

**Mistake 1: A weak docstring.**
If your tool says `"""Does stuff."""`, the model has no idea when to use it. Write the docstring as if you're explaining the tool to a new colleague: what it does, and when to use it.

**Mistake 2: Forgetting type hints.**
`def get_weather(city):` without `: str` gives the model less information about what to send. Always add type hints to tool arguments.

**Mistake 3: Naming the tool node something other than "tools" with `tools_condition`.**
`tools_condition` routes to a node called `"tools"`. If you named yours `"my_tools"`, the route silently goes nowhere. Either name it `"tools"`, or write your own router like we did in section 4.

**Mistake 4: Forgetting the edge back to the agent.**
If you leave out `builder.add_edge("tools", "agent")`, the tools run, and then the graph has nowhere to go, so it just stops, and you never get the final answer. Let's see what that looks like:

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
# builder.add_edge("tools", "agent")      <-- forgot this line

In [ ]:
broken_agent = builder.compile()

In [ ]:
result = broken_agent.invoke({"messages": [("user", "What is 6 times 7?")]})

# The last message is the raw tool output, not a friendly answer from the model
print("Last message type:", result["messages"][-1].type)
print("Last message:", result["messages"][-1].content)

## 9. Mini project: a travel helper agent

Three tools: weather, currency conversion and a packing tip. To keep this free and predictable, the tools return made-up data. In a real app they'd call a weather API and an exchange-rate API, and nothing else in the agent would change.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get today's weather for a city."""
    weather_data = {
        "mumbai": "31 C, humid, light rain",
        "delhi": "36 C, sunny",
        "bengaluru": "24 C, pleasant",
        "london": "14 C, cloudy",
    }
    return weather_data.get(city.lower(), f"No weather data for {city}")

In [ ]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between INR, USD, EUR and GBP."""
    # How many rupees one unit of each currency is worth (example rates)
    rupee_value = {"INR": 1, "USD": 83, "EUR": 90, "GBP": 105}

    source = from_currency.upper()
    target = to_currency.upper()
    if source not in rupee_value or target not in rupee_value:
        return "Sorry, I only know INR, USD, EUR and GBP."

    converted = amount * rupee_value[source] / rupee_value[target]
    return f"{amount} {source} = {round(converted, 2)} {target}"

In [ ]:
@tool
def packing_tip(weather: str) -> str:
    """Suggest what to pack, based on a weather description."""
    weather = weather.lower()
    if "rain" in weather:
        return "Pack an umbrella and clothes that dry quickly."
    if "sunny" in weather:
        return "Pack sunscreen, sunglasses and a water bottle."
    if "cloud" in weather:
        return "Pack a light jacket."
    return "Pack comfortable clothes."

In [ ]:
travel_tools = [get_weather, convert_currency, packing_tip]
travel_llm = llm.bind_tools(travel_tools)

TRAVEL_SYSTEM = SystemMessage(content=(
    "You are a friendly travel assistant. Use the tools. Never guess the weather or exchange rates."
))

In [ ]:
def travel_agent(state: MessagesState) -> dict:
    reply = travel_llm.invoke([TRAVEL_SYSTEM] + state["messages"])
    return {"messages": [reply]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", travel_agent)
builder.add_node("tools", ToolNode(travel_tools))

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

In [ ]:
travel = builder.compile()

In [ ]:
question = ("I'm going to London with 50000 INR. What's the weather, "
            "what should I pack, and how many pounds is my money?")

result = travel.invoke({"messages": [("user", question)]})

for m in result["messages"]:
    if m.type == "ai" and m.tool_calls:
        print("Asked for:", [call["name"] for call in m.tool_calls])

print()
print(result["messages"][-1].content)

### Your turn

Add a fourth tool, `get_famous_food(city: str) -> str`, with a small dictionary of cities and dishes. Add it to `travel_tools`, rebuild the agent, and ask "What should I eat in Mumbai?". Starter code is below.

In [ ]:
@tool
def get_famous_food(city: str) -> str:
    """Get a famous local food for a city."""
    foods = {"mumbai": "vada pav", "delhi": "chole bhature", "kolkata": "rosogolla"}   # add more
    return foods.get(city.lower(), "I don't know the food there yet.")

In [ ]:
my_tools = travel_tools + [get_famous_food]

my_agent = create_agent(model=llm, tools=my_tools, system_prompt="You are a friendly travel assistant. Use the tools.")

In [ ]:
result = my_agent.invoke({"messages": [("user", "What should I eat in Mumbai?")]})
print(result["messages"][-1].content)

## 10. Practice

**Exercise 1.** Write a tool `word_count(text: str) -> int` and give it to `create_agent`. Ask: "How many words are in 'LangGraph makes agents easy to build'?"

**Exercise 2.** Add a safety limit to the hand-built agent. Extend the state with `rounds: int`, add 1 every time the agent node runs, and make the router stop after 3 rounds even if the model still wants tools. (Remember Day 3: never trust a loop without a way out.)

In [ ]:
# Solution 1
@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

In [ ]:
counting_agent = create_agent(model=llm, tools=[word_count])

result = counting_agent.invoke(
    {"messages": [("user", "How many words are in 'LangGraph makes agents easy to build'?")]}
)
print(result["messages"][-1].content)

In [ ]:
# Solution 2
class SafeState(MessagesState):
    rounds: int

In [ ]:
def counted_agent(state: SafeState) -> dict:
    reply = llm_with_tools.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply], "rounds": state.get("rounds", 0) + 1}

In [ ]:
def safe_route(state: SafeState) -> Literal["tools", "__end__"]:
    if state["rounds"] >= 3:
        return END                       # enough, stop no matter what
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
builder = StateGraph(SafeState)

In [ ]:
builder.add_node("agent", counted_agent)
builder.add_node("tools", ToolNode(tools))

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", safe_route)
builder.add_edge("tools", "agent")

In [ ]:
safe_agent = builder.compile()

In [ ]:
result = safe_agent.invoke({"messages": [("user", "Multiply 6 by 7")], "rounds": 0})
print("rounds used:", result["rounds"])
print(result["messages"][-1].content)

## Wrapping up

A tool is a Python function with `@tool`, and the model decides when to use it based on the **name, type hints and docstring**. `bind_tools` tells the model what's available. The model only *asks*; your code runs the tool and replies with a `ToolMessage`.

An agent is that ask-run-reply cycle in a loop, which is exactly the Day 3 loop pattern with an LLM inside. `ToolNode` and `tools_condition` are the ready-made pieces, and `create_agent` builds the whole thing in one call.

There's one problem left. Every time you call your agent, it starts fresh. Tell it your name, call it again, and it has forgotten. Tomorrow we fix that with **memory**.